# Prompts

In [ ]:
import os
import warnings
from pathlib import Path
from typing import Any, Generator, Type, TypeVar

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str: str = "google/gemini-2.0-flash-001"

# Deterministic responses
llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str,
)

# Creative responses
creative_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.9,
    model=model_str,
)

<br>

### Basic Prompting

- We'll start by looking at the various parts of our prompt.

- For RAG use-cases we'll typically have three core components however this is very use-cases dependant and can vary significantly.

- Nonetheless, for RAG we will typically see:
    - **Rules for our LLM**: this part of the prompt sets up the behavior of our LLM, how it should approach responding to user queries, and simply providing as much information as possible about what we're wanting to do as possible. We typically place this within the system prompt of an chat LLM.

    - **Context**: this part is RAG-specific. The context refers to some external information that we may have retrieved from a web search, database query, or often a vector database. This external information is the Retrieval Augmentation part of RAG. For chat LLMs we'll typically place this inside the chat messages between the assistant and user.

    - **Question**: this is the input from our user. In the vast majority of cases the question/query/user input will always be provided to the LLM (and typically through a user message). However, the format and location of this being provided often changes.

    - **Answer**: this is the answer from our assistant, again this is very typical and we'd expect this with every use-case.

Below is an example of how a RAG prompt may look:

```txt
Answer the question based on the context below,                 }
if you cannot answer the question using the                     }--->  (Rules) For Our Prompt
provided information answer with "I don't know"                 }

Context: Aurelio AI is an AI development studio                 }
focused on the fields of Natural Language Processing (NLP)      }
and information retrieval using modern tooling                  }--->   Context AI has
such as Large Language Models (LLMs),                           }
vector databases, and LangChain.                                }

Question: Does Aurelio AI do anything related to LangChain?     }--->   User Question

Answer:                                                         }--->   AI Answer
```

<br>

- Here we can see how the AI will appoach our question, as you can see we have a formulated response, if the context has the answer, then use the context to answer the question, if not, say I don't know, then we also have context and question which are being passed into this similarly to paramaters in a function.

In [5]:
from langchain.prompts import ChatPromptTemplate

prompt: str = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

<context>
{context}
</context>
"""

prompt_template: ChatPromptTemplate = ChatPromptTemplate.from_messages(
    [
        ("system", prompt),
        ("user", "{query}"),
    ]
)

- When we call the template it will expect us to provide two variables, the context and the query.
- Both of these variables are pulled from the strings we wrote, as LangChain interprets curly-bracket syntax (ie `{context}` and `{query}`) as indicating a dynamic variable that we expect to be inserted at query time.
- We can see that these variables have been picked up by our template object by viewing it's input_variables attribute:

In [6]:
prompt_template.input_variables

['context', 'query']

In [8]:
console.print(prompt_template.messages)

[
    SystemMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=['context'],
            input_types={},
            partial_variables={},
            template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question 
using the\nprovided information answer with "I don\'t know".\n\n<context>\n{context}\n</context>\n'
        ),
        additional_kwargs={}
    ),
    HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=['query'],
            input_types={},
            partial_variables={},
            template='{query}'
        ),
        additional_kwargs={}
    )
]

<br>

### Invoking our LLM with Templates

- We've defined our prompt template, now let's define out LLM and run it with our template and a user query.

In [9]:
pipeline = (
    {
        "query": lambda x: x["query"],
        "context": lambda x: x["context"],
    }
    | prompt_template
    | llm
    | {"response": lambda x: x.content}
)

In [10]:
context: str = """
Goalkeeper James Trafford has rejoined Manchester City from Burnley in a deal which the Clarets say is a new record 
fee for a British goalkeeper.

City sources say the move is worth £27m plus add-ons.

However, Burnley sources put the figure at £31m plus add-ons and a sell-on clause.

That would take the deal above the previous record of £30m which Everton paid Sunderland for Jordan Pickford in 2017.

City academy graduate Trafford, 22, signed for the Clarets for up to £19m in July 2023.

He impressed last season as Scott Parker's side won promotion back to the Premier League - keeping 29 clean sheets 
across 45 Championship games - and was named in the division's team of the year.

Announcing his departure, Burnley said "the move to City is a record transfer fee for a British goalkeeper and a club 
record sale".

City had a buy-back clause for Trafford and also matching rights, allowing them to equal any offer from another club.

They moved for the player following a £27m bid from Newcastle - and Trafford opted for a return to Manchester.

He has signed a five-year contract with the option for another year at Etihad Stadium, and will wear the number one 
shirt.
"""

In [11]:
query = "What is the record fee for a British goalkeeper?"

response = pipeline.invoke({"context": context, "query": query})
console.print(response)

{
    'response': "According to Burnley sources, James Trafford's move to Manchester City is a record transfer fee 
for a British goalkeeper at £31m plus add-ons and a sell-on clause. This would exceed the previous record of £30m 
paid by Everton for Jordan Pickford in 2017.\n"
}

### Few Shot Prompting

- Many `State-of-the-Art (SotA)` LLMs are incredible at instruction following.

- Meaning that it requires much less effort to get the intended output or behavior from these models than is the case for older LLMs and smaller LLMs.

- Before creating an example let's first see how to use LangChain's few shot prompting objects.

- We will provide multiple examples and we'll feed them in as sequential human and ai messages so we setup the template like this:

In [12]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

examples = [
    {"input": "Here is query #1", "output": "Here is the answer #1"},
    {"input": "Here is query #2", "output": "Here is the answer #2"},
    {"input": "Here is query #3", "output": "Here is the answer #3"},
]

In [13]:
from langchain.prompts import FewShotChatMessagePromptTemplate

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
console.print(few_shot_prompt.format())

Human: Here is query #1
AI: Here is the answer #1
Human: Here is query #2
AI: Here is the answer #2
Human: Here is query #3
AI: Here is the answer #3

<br>

### Few-Shot Example

- Using a tiny LLM limits it's ability, so when asking for specific behaviors or structured outputs it can struggle.
- For example, we'll ask the LLM to summarize the key points about Aurelio AI using markdown and bullet points.
- Let's see what happens.

In [17]:
console.print(prompt_template.messages[0].prompt.template)

Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

<context>
{context}
</context>

In [19]:
new_system_prompt: str = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Always answer in markdown format. When doing so please
provide headers, short summaries, follow with bullet
points, then conclude.

<context>
{context}
</context>
"""

# Update the system prompt in the template
prompt_template.messages[0].prompt.template = new_system_prompt

out = pipeline.invoke({"query": query, "context": context})
console.print(out.response)

{
    'response': "## Record Fee for a British Goalkeeper\n\n**Summary:**\n\nJames Trafford's recent transfer back to
Manchester City from Burnley has potentially set a new record fee for a British goalkeeper. While sources differ on
the exact amount, Burnley claims the fee surpasses the previous record held by Jordan Pickford.\n\n**Key 
points:**\n\n*   Burnley claims the transfer fee for James Trafford is a new record for a British goalkeeper.\n*   
Burnley sources put the figure at £31m plus add-ons and a sell-on clause.\n*   The previous record was £30m, paid 
by Everton for Jordan Pickford in 2017.\n\n**Conclusion:**\n\nBased on Burnley's statement, James Trafford's 
transfer fee exceeds the previous record for a British goalkeeper, although the exact figure is subject to 
conflicting reports.\n"
}

In [21]:
console.print(out["response"])

## Record Fee for a British Goalkeeper

**Summary:**

James Trafford's recent transfer back to Manchester City from Burnley has potentially set a new record fee for a 
British goalkeeper. While sources differ on the exact amount, Burnley claims the fee surpasses the previous record 
held by Jordan Pickford.

**Key points:**

*   Burnley claims the transfer fee for James Trafford is a new record for a British goalkeeper.
*   Burnley sources put the figure at £31m plus add-ons and a sell-on clause.
*   The previous record was £30m, paid by Everton for Jordan Pickford in 2017.

**Conclusion:**

Based on Burnley's statement, James Trafford's transfer fee exceeds the previous record for a British goalkeeper, 
although the exact figure is subject to conflicting reports.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(out["response"]))

## Record Fee for a British Goalkeeper

**Summary:**

James Trafford's recent transfer back to Manchester City from Burnley has potentially set a new record fee for a British goalkeeper. While sources differ on the exact amount, Burnley claims the fee surpasses the previous record held by Jordan Pickford.

**Key points:**

*   Burnley claims the transfer fee for James Trafford is a new record for a British goalkeeper.
*   Burnley sources put the figure at £31m plus add-ons and a sell-on clause.
*   The previous record was £30m, paid by Everton for Jordan Pickford in 2017.

**Conclusion:**

Based on Burnley's statement, James Trafford's transfer fee exceeds the previous record for a British goalkeeper, although the exact figure is subject to conflicting reports.


### Chain of Thought Prompting

- We'll take a look at one more commonly used prompting technique called chain of thought (CoT).

- CoT is a technique that encourages the LLM to think through the problem step by step before providing an answer.

- The idea being that by breaking down the problem into smaller steps, the LLM is more likely to arrive at the correct answer and we are less likely to see hallucinations.

- To implement CoT we don't need any specific LangChain objects, instead we are simply modifying how we instruct our LLM within the system prompt.

- We will ask the LLM to list the problems that need to be solved, to solve each problem individually, and then to arrive at the final answer.


In [24]:
no_cot_system_prompt: str = """
You're a helpful assistant that answers the user's question.

You MUST answer the question directly without any other
text or explanation.
"""

no_cot_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", no_cot_system_prompt),
        ("user", "{query}"),
    ]
)

- Nowadays most LLMs are trained to use CoT prompting by default, so we actually need to instruct it not to do so for this example which is why we added `"You MUST answer the question directly without any other text or explanation."` to our system prompt.

In [ ]:
query = "How many keystrokes are needed to type the numbers from 1 to 500?"

no_cot_pipeline = no_cot_prompt_template | llm | {"response": lambda x: x.content}
no_cot_result = no_cot_pipeline.invoke({"query": query})
console.print(no_cot_result)

{'response': '1392\n'}

- The actual answer is 1392.

- The LLM got this correctly probably because LLM has been trained to answer such questions.

In [28]:
# Define the chain-of-thought prompt template
cot_system_prompt = """
You're a helpful assistant that answers the user's question.

To answer the question, you must:

- List systematically and in precise detail all
  subproblems that need to be solved to answer the
  question.
- Solve each sub problem INDIVIDUALLY and in sequence.
- Finally, use everything you have worked through to
  provide the final answer.
"""

cot_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", cot_system_prompt),
        ("user", "{query}"),
    ]
)
cot_pipeline = cot_prompt_template | llm | {"response": lambda x: x.content}

cot_result = cot_pipeline.invoke({"query": query})
Markdown(cot_result["response"])

Here's how we can solve this problem:

**1. Break down the problem into categories based on the number of digits:**

*   **1-digit numbers:** 1 to 9
*   **2-digit numbers:** 10 to 99
*   **3-digit numbers:** 100 to 500

**2. Calculate the number of keystrokes for each category:**

*   **1-digit numbers (1 to 9):**
    *   There are 9 numbers.
    *   Each number requires 1 keystroke.
    *   Total keystrokes: 9 * 1 = 9

*   **2-digit numbers (10 to 99):**
    *   There are 99 - 10 + 1 = 90 numbers.
    *   Each number requires 2 keystrokes.
    *   Total keystrokes: 90 * 2 = 180

*   **3-digit numbers (100 to 500):**
    *   There are 500 - 100 + 1 = 401 numbers.
    *   Each number requires 3 keystrokes.
    *   Total keystrokes: 401 * 3 = 1203

**3. Sum the keystrokes from each category:**

*   Total keystrokes = Keystrokes (1-digit) + Keystrokes (2-digit) + Keystrokes (3-digit)
*   Total keystrokes = 9 + 180 + 1203 = 1392

**Answer:** It takes 1392 keystrokes to type the numbers from 1 to 500.
